# Emergent Expansion Cosmology — EEC v10
**Ismail Khan**
ismailkhan.mcs@gmail.com

ORCID: 0009-0005-7485-88

https://doi.org/10.5281/zenodo.21412819

Copyright 2026 Ismail Khan. All rights reserved.

Scale-free formulation: the stiffness is kappa = beta*rho_m (no characteristic density, no exponential cutoff), with beta fixed by flatness. rho_neq(z) = (1-Om-Or)(1+z)^3 D^4(z)/D^4(0). Three free parameters: omega_m, H0, sigma_8.


**v10:** adds the exact late-time attractor cell (f* = (√73−1)/12, w* = (1−√73)/9, q* = 2/3−√73/6; H → 0, Ω_neq → 1), computed with the full self-consistent engine extended past a = 1. Fit, datasets, and all v8/v9 numerical results unchanged.


In [ ]:
# Cell 1 — Setup
import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','emcee','corner','-q'])
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CKPT_DIR='/content/drive/MyDrive/EEC_V10'
except Exception:
    CKPT_DIR='ckpt_EEC_V10'
os.makedirs(CKPT_DIR, exist_ok=True)
import emcee, numpy as np
print(f'emcee {emcee.__version__} | checkpoints → {CKPT_DIR}')


Mounted at /content/drive
emcee 3.1.6 | checkpoints → /content/drive/MyDrive/EEC_V10


In [ ]:
# Cell 2 — Physical constants and fixed quantities
import warnings
warnings.filterwarnings('ignore')

c_km      = 299792.458         # km/s
omega_gam = 2.469e-5           # photon density
N_eff     = 3.046
omega_nu  = (7/8)*(4/11)**(4/3)*N_eff*omega_gam
omega_r   = omega_gam + omega_nu   # total radiation

# ── omega_b: fixed to BBN, model-independent ──────────────────────────────
# Cooke, Pettini & Steidel 2016, ApJ 830, 148
# omega_b h^2 = 0.02229 +/- 0.00036 from primordial deuterium D/H.
# Pre-recombination physics — EEC contribution is exactly zero there.
# BBN sigma = 0.00036 shifts theta* by only 0.05 sigma => safe to fix.
omega_b = 0.02233

# ── Planck theta* (model-independent) ─────────────────────────────────────
# Planck 2018 parameter tables (Planck Collaboration 2020, A&A 641, A6)
# 100*theta_MC = 1.0409 +/- 0.0006 — model-independent acoustic scale
# (measured directly from CMB peak positions, no LCDM geometry assumed)
# sigma = 0.000006 is the correct model-independent value from Table 2.
# Previous notebooks used 0.000031 (5x too conservative) — corrected here.
THETA_STAR_OBS = 0.010409
THETA_STAR_ERR = 0.000006

# ── EEC scale-free formulation (scale-free, 3-parameter) ───────────────────────────────
# No characteristic density. Coleman-Gurtin admissibility + scale-freedom fix
# the stiffness uniquely as kappa = beta * rho_m, with beta set by flatness.
# rho_neq(z) = (1 - Om - Or) * (1+z)^3 * D^4(z)/D^4(0).  No z_c, no exp cutoff.
SCALE_FREE   = True

# ── MCMC configuration ────────────────────────────────────────────────────
N_WALKERS  = 32
N_BURN     = 600
N_PROD     = 3000
BATCH      = 2
DISP_SECS  = 30
CKPT_SECS  = 300

# Parameter bounds (3 free params)
OM_LO, OM_HI = 0.08, 0.35
H0_LO, H0_HI = 40., 100.
S8_LO, S8_HI = 0.4, 1.2

# Growth solver config
GROWTH_NPTS = 300
GROWTH_ZMAX = 50
EEC_NITER   = 3
CONV_THR    = 1e-6

print(f'omega_b  = {omega_b} (BBN, Cooke+2016 ApJ 830 148, fixed)')
print(f'theta*   = {THETA_STAR_OBS} +/- {THETA_STAR_ERR} (Planck, model-independent)')
print(f'Scale-free formulation: kappa = beta*rho_m (no characteristic density)')
print(f'Free params: omega_m, H0, sigma_8  (ndim=3, same as LCDM)')



omega_b  = 0.02233 (BBN, Cooke+2016 ApJ 830 148, fixed)
theta*   = 0.010409 +/- 6e-06 (Planck, model-independent)
Scale-free formulation: kappa = beta*rho_m (no characteristic density)
Free params: omega_m, H0, sigma_8  (ndim=3, same as LCDM)


In [ ]:
# Cell 3 — Pantheon+
import urllib.request
BASE=('https://raw.githubusercontent.com/PantheonPlusSH0ES/'
      'DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/')
DAT='Pantheon+SH0ES.dat'; COV='Pantheon+SH0ES_STAT+SYS.cov'
for fn,url in [(DAT,BASE+DAT),(COV,BASE+COV)]:
    if not os.path.exists(fn):
        print(f'Downloading {fn}...', end=' ', flush=True)
        urllib.request.urlretrieve(url, fn)
        print('done')
pp   = np.genfromtxt(DAT, names=True, dtype=None, encoding=None)
mask = (pp['IS_CALIBRATOR']==0) & (pp['zHD'].astype(float)>0.01)
z_sn = pp['zHD'][mask].astype(float)
mb   = pp['m_b_corr'][mask].astype(float)
good = np.isfinite(z_sn) & np.isfinite(mb)
z_sn, mb = z_sn[good], mb[good]
idx  = np.argsort(z_sn); z_sn, mb = z_sn[idx], mb[idx]
with open(COV) as f:
    n_cov = int(f.readline().strip())
    cov_flat = np.loadtxt(f)
C_sn_full = cov_flat.reshape(n_cov,n_cov)
fi   = np.where(mask)[0][good][np.argsort(pp['zHD'][mask].astype(float)[good])]
C_sn = C_sn_full[np.ix_(fi,fi)]
from scipy.linalg import cholesky
L_sn = cholesky(C_sn, lower=True)
N_SN = len(z_sn)
print(f'Pantheon+: {N_SN} SNe Ia  (z={z_sn[0]:.3f}–{z_sn[-1]:.3f})')


Pantheon+: 1580 SNe Ia  (z=0.010–2.261)


In [ ]:
# Cell 4 — BOSS DR12 + DESI DR1
# ── BOSS DR12 (Alam+2017, MNRAS 470, 2617, Table 3) ──────────────────────
BOSS_Z   = np.array([0.38, 0.51, 0.61])
BOSS_DM  = np.array([10.27, 13.38, 15.45])   # DM/rd
BOSS_DH  = np.array([25.00, 22.33, 20.99])   # DH/rd
BOSS_FS8 = np.array([0.497, 0.458, 0.436])
BOSS_SDM = np.array([0.15, 0.19, 0.22])
BOSS_SDH = np.array([0.53, 0.38, 0.35])
BOSS_SFS8= np.array([0.045, 0.038, 0.034])
BOSS_R_DM_DH  = np.array([-0.47, -0.43, -0.43])  # DM-DH correlation
BOSS_R_DM_FS8 = np.array([ 0.31,  0.29,  0.28])  # DM-fσ8 correlation (Alam+2017 Table 3)
BOSS_R_DH_FS8 = np.array([-0.31, -0.27, -0.26])  # DH-fσ8 correlation (Alam+2017 Table 3)
BOSS_OBS = np.concatenate([[BOSS_DM[i],BOSS_DH[i],BOSS_FS8[i]] for i in range(3)])
def _boss_cov():
    # Full 9x9 covariance with all cross-correlations (Alam+2017 Table 3)
    C=np.zeros((9,9))
    for i in range(3):
        j=i*3
        # Diagonal variances
        C[j,j]  =BOSS_SDM[i]**2
        C[j+1,j+1]=BOSS_SDH[i]**2
        C[j+2,j+2]=BOSS_SFS8[i]**2
        # DM-DH cross-correlation
        C[j,j+1]=C[j+1,j]=BOSS_R_DM_DH[i]*BOSS_SDM[i]*BOSS_SDH[i]
        # DM-fσ8 cross-correlation
        C[j,j+2]=C[j+2,j]=BOSS_R_DM_FS8[i]*BOSS_SDM[i]*BOSS_SFS8[i]
        # DH-fσ8 cross-correlation
        C[j+1,j+2]=C[j+2,j+1]=BOSS_R_DH_FS8[i]*BOSS_SDH[i]*BOSS_SFS8[i]
    return C
BOSS_ICOV = np.linalg.inv(_boss_cov())

# ── DESI DR1 (DESI Collaboration 2024, arXiv:2404.03002, Table 1) ─────────
DESI_ISO_Z   = np.array([0.295, 1.491])          # BGS, QSO
DESI_ISO_DV  = np.array([7.93,  26.07])
DESI_ISO_SDV = np.array([0.15,   0.67])
DESI_ANI_Z   = np.array([0.510, 0.706, 0.930, 1.317, 2.330])
DESI_ANI_DM  = np.array([13.62, 16.85, 21.71, 27.79, 39.71])
DESI_ANI_DH  = np.array([20.98, 20.08, 17.88, 13.82,  8.52])
DESI_ANI_SDM = np.array([0.25,   0.32,  0.28,  0.69,  0.94])
DESI_ANI_SDH = np.array([0.61,   0.60,  0.35,  0.42,  0.17])
DESI_ANI_R   = np.array([-0.445,-0.420,-0.389,-0.444,-0.477])

print('BOSS DR12: 3 bins (z=0.38, 0.51, 0.61) — BAO + fsig8')
print('DESI DR1:  7 bins (z=0.295–2.330) — BGS,LRG,ELG,QSO,Lyα')


BOSS DR12: 3 bins (z=0.38, 0.51, 0.61) — BAO + fsig8
DESI DR1:  7 bins (z=0.295–2.330) — BGS,LRG,ELG,QSO,Lyα


In [ ]:
# Cell 5 — EEC Engine
from scipy.integrate import quad, solve_ivp, cumulative_trapezoid
from scipy.linalg import solve_triangular

# ── Eisenstein & Hu 1998 fitting formulae ─────────────────────────────────
def zdrag(omm):
    """Baryon drag epoch (E&H 1998 Eq 4). omega_b fixed to BBN."""
    b1 = 0.313*omm**(-0.419)*(1 + 0.607*omm**0.674)
    b2 = 0.238*omm**0.223
    return 1291.*omm**0.251/(1 + 0.659*omm**0.828)*(1 + b1*omega_b**b2)

def zstar(omm):
    """Photon decoupling epoch (Hu & Sugiyama 1996). omega_b fixed to BBN."""
    g1 = 0.0783*omega_b**(-0.238)/(1 + 39.5*omega_b**0.763)
    g2 = 0.560/(1 + 21.1*omega_b**1.81)
    return 1048.*(1 + 0.00124*omega_b**(-0.738))*(1 + g1*omm**g2)

# ── Sound horizon ──────────────────────────────────────────────────────────
# Uses ONLY matter+radiation H(z) — correct because:
# EEC contribution ~ (1+z)^3 D^4 -> ~10^-11 at z~300 (growth-suppressed)
# => EEC is negligible at recombination => LCDM H(z) is exact here
def rs_phys_Mpc(omm, H0, z_end=None):
    """Sound horizon in Mpc. z_end: drag epoch (default) or z*."""
    if z_end is None:
        z_end = zdrag(omm)
    h = H0/100.; Om = omm/h**2; Or = omega_r/h**2
    def integrand(z):
        R  = 3.*omega_b/(4.*omega_gam*(1. + z))
        cs = c_km/np.sqrt(3.*(1. + R))
        Hz = H0*np.sqrt(Om*(1+z)**3 + Or*(1+z)**4)
        return cs/Hz
    v, _ = quad(integrand, z_end, 1e6, limit=100, epsabs=1e-8, epsrel=1e-8)
    return v

def rd(omm, H0):
    return rs_phys_Mpc(omm, H0, zdrag(omm))

def rs_star(omm, H0):
    return rs_phys_Mpc(omm, H0, zstar(omm))

# ── EEC growth solver ──────────────────────────────────────────────────────
def _beta_norm(Om, Or, s8):
    # flatness: beta = 2(1-Om-Or)/(Om*s8^4); rho_neq normalized to (1-Om-Or) at z=0
    d = Om*s8**4
    return 1e10 if d < 1e-30 else 2.*(1. - Om - Or)/d

def _solve_growth(Om, Or, Ef, mu=None):
    if mu is None:
        mu = lambda z: 1.
    a0   = 1./(1. + GROWTH_ZMAX)
    a_arr = np.linspace(a0, 1., GROWTH_NPTS)
    def rhs(a, y):
        z  = 1./a - 1.
        Ev = Ef(z)
        dz = max(z*1e-5, 1e-6)
        dEdz = ((Ef(z+dz)-Ef(max(z-dz,0.)))/(2*dz) if z > dz
                else (Ef(dz)-Ef(0.))/dz)
        c1 = 3./a - dEdz/(a**2*Ev)
        c0 = -1.5*Om*mu(z)/(a**5*Ev**2)
        return [y[1], -c1*y[1] - c0*y[0]]
    sol = solve_ivp(rhs, [a0,1.], [a0,1.], t_eval=a_arr,
                    method='RK45', rtol=1e-7, atol=1e-9)
    D   = sol.y[0]/sol.y[0,-1]
    z_a = (1./a_arr-1.)[::-1]
    return lambda z: float(np.interp(z, z_a, D[::-1]))

def iterate_EEC(omm, s8, H0):
    """Self-consistent scale-free EEC solver. Returns (Dc, beta, Ef, Om, Or, None)."""
    h   = H0/100.
    Om  = omm/h**2
    Or  = omega_r/h**2
    beta = _beta_norm(Om, Or, s8)
    Onq0 = 1. - Om - Or                      # flatness: Omega_neq(0)
    def make_E(Dc):
        D0 = Dc(0.)
        D04 = D0**4 if D0 > 1e-30 else 1e-30
        def Ef(z):
            Oneq = Onq0*(1+z)**3*Dc(z)**4/D04
            return np.sqrt(max(Om*(1+z)**3 + Or*(1+z)**4 + Oneq, 1e-30))
        return Ef
    E0 = lambda z: np.sqrt(Om*(1+z)**3 + Or*(1+z)**4)
    Dc = lambda z, _D=_solve_growth(Om, Or, E0): s8*_D(z)
    for _ in range(EEC_NITER):
        _Dc = Dc
        Ec  = make_E(_Dc)
        D0  = _Dc(0.); D04 = D0**4 if D0 > 1e-30 else 1e-30
        def mu(z, _D=_Dc, _D04=D04):
            Oneq = Onq0*(1+z)**3*_D(z)**4/_D04
            Omz  = Om*(1+z)**3
            return 1. if Omz < 1e-30 else 1. + Oneq/Omz
        Dc_new = lambda z, _D=_solve_growth(Om, Or, Ec, mu): s8*_D(z)
        diff   = max(abs(Dc_new(z)-_Dc(z)) for z in np.linspace(0,10,80))
        Dc     = Dc_new
        if diff < CONV_THR:
            break
    return Dc, beta, make_E(Dc), Om, Or, None

# ── Comoving distance using EEC E(z) ──────────────────────────────────────
def chi_EEC(z_up, Ef, H0):
    """chi(z_up) in Mpc using self-consistent EEC E(z)."""
    pts = [p for p in [0.5,2,10,100,500,1000] if p < z_up]
    v, _ = quad(lambda z: c_km/(H0*Ef(z)), 0, z_up,
                limit=100, points=pts)
    return v

# ── theta* — THE KEY FUNCTION ──────────────────────────────────────────────
def compute_theta_star(omm, H0, Ef):
    """
    theta* = r_s(z*) / chi(z*)

    r_s uses:  LCDM-only H(z) (pre-recombination, EEC=0 there)
               omega_b = 0.02233 (BBN, model-independent, NOT Planck)
    chi uses:  EEC E(z) (self-consistent, EEC geometry)

    This is model-independent: r_s is fixed by physics before recombination
    where EEC is exactly suppressed; chi encodes the EEC expansion history.
    """
    zs  = zstar(omm)
    rs  = rs_star(omm, H0)
    chi = chi_EEC(zs, Ef, H0)
    return rs/chi

def fsig8(z, Dc, dz=0.01):
    """f*sigma8(z) = -(1+z) * d[Dc]/dz  where Dc(z)=sigma8*D(z)/D(0)."""
    D0 = Dc(z)
    if D0 < 1e-10: return 0.
    dDdz = ((Dc(z+dz)-Dc(max(z-dz,0.)))/(2*dz) if z > dz
            else (Dc(dz)-Dc(0.))/dz)
    return -(1.+z)*dDdz

# quick engine test
_Dc,_,_Ef,_Om,_Or,_ = iterate_EEC(0.143, 0.77, 69.)
_th = compute_theta_star(0.143, 69., _Ef)
print(f'Engine test (scale-free): Om={_Om:.4f}  theta*={_th:.6f}'
      f'  ({(_th-THETA_STAR_OBS)/THETA_STAR_ERR:+.1f}σ from Planck)')


Engine test (scale-free): Om=0.3004  theta*=0.010322  (-14.6σ from Planck)


In [ ]:
# Cell 6 — Likelihood
# 3 free parameters: omega_m, H0, sigma_8
# Planck theta* replaces Planck omega_m prior (see header)

def _chi2_decomposed(omm, H0, s8):
    """Returns dict of chi2 contributions. Returns None on error."""
    h  = H0/100.
    Om = omm/h**2
    Or = omega_r/h**2
    if Om+Or >= 1. or Om <= 0:
        return None
    try:
        Dc, beta, Ef, Om, Or, _ = iterate_EEC(omm, s8, H0)
        if abs(Ef(0.) - 1.) > 0.02:
            return None

        parts = {}

        # ── (1) Planck theta* ───────────────────────────────────────────
        # r_s(z*): pre-recombination physics with BBN omega_b (model-indep)
        # chi(z*): EEC geometry (self-consistent)
        th_eec          = compute_theta_star(omm, H0, Ef)
        parts['th_val'] = th_eec
        parts['th_sig'] = (th_eec - THETA_STAR_OBS)/THETA_STAR_ERR
        parts['theta']  = parts['th_sig']**2

        chi2 = parts['theta']

        # ── (2) Pantheon+ SNe (analytic M marginalisation) ──────────────
        z_max  = max(z_sn[-1]*1.01, 2.6)
        z_fine = np.linspace(0, z_max, 2000)
        E_fine = np.array([Ef(z) for z in z_fine])
        chi_cum = np.concatenate([[0.],
            cumulative_trapezoid(c_km/(H0*E_fine), z_fine)])

        dL   = (1+z_sn)*np.interp(z_sn, z_fine, chi_cum)
        mu_th = 5.*np.log10(np.maximum(dL,1e-10)) + 25.
        delta = mb - mu_th
        ones  = np.ones_like(delta)
        y1 = solve_triangular(L_sn, delta, lower=True)
        y2 = solve_triangular(L_sn, ones,  lower=True)
        Cd = solve_triangular(L_sn.T, y1, lower=False)
        C1 = solve_triangular(L_sn.T, y2, lower=False)
        A_ = float(delta@Cd); B_ = float(delta@C1); D_ = float(ones@C1)
        parts['sn']  = A_ - B_**2/D_
        chi2        += parts['sn']

        # ── (3) BOSS DR12 (BAO + fsig8) ─────────────────────────────────
        rd_ = rd(omm, H0)
        DM_b  = np.array([float(np.interp(z,z_fine,chi_cum))/rd_ for z in BOSS_Z])
        DH_b  = c_km/(H0*np.interp(BOSS_Z,z_fine,E_fine))/rd_
        F8_b  = np.array([fsig8(z,Dc) for z in BOSS_Z])
        theory= np.concatenate([[DM_b[i],DH_b[i],F8_b[i]] for i in range(3)])
        dv    = BOSS_OBS - theory
        parts['boss'] = float(dv@BOSS_ICOV@dv)
        chi2         += parts['boss']

        # ── (4) DESI isotropic ───────────────────────────────────────────
        c2di = 0.
        for i in range(len(DESI_ISO_Z)):
            z_  = DESI_ISO_Z[i]
            DM_ = float(np.interp(z_,z_fine,chi_cum))
            DH_ = c_km/(H0*float(np.interp(z_,z_fine,E_fine)))
            DV_ = (z_*DH_*DM_**2)**(1./3.)/rd_
            c2di += ((DESI_ISO_DV[i]-DV_)/DESI_ISO_SDV[i])**2
        parts['desi_iso'] = c2di
        chi2             += c2di

        # ── (5) DESI anisotropic ─────────────────────────────────────────
        c2da = 0.
        for i in range(len(DESI_ANI_Z)):
            z_ = DESI_ANI_Z[i]
            DM_= float(np.interp(z_,z_fine,chi_cum))/rd_
            DH_= c_km/(H0*float(np.interp(z_,z_fine,E_fine)))/rd_
            dM = DESI_ANI_DM[i]-DM_; dH = DESI_ANI_DH[i]-DH_
            r  = DESI_ANI_R[i]
            s1 = DESI_ANI_SDM[i]; s2 = DESI_ANI_SDH[i]
            det= (1-r**2)*s1**2*s2**2
            c2da += (dM**2*s2**2 - 2*r*s1*s2*dM*dH + dH**2*s1**2)/det
        parts['desi_ani'] = c2da
        chi2             += c2da

        # S8 reported as a derived quantity (sigma8 constrained by RSD/fsig8; no lensing in fit)
        # S8 = sigma8 * sqrt(Om/0.3); Om is the engine-derived matter fraction.
        S8_model = s8*np.sqrt(Om/0.3)
        parts['S8_model'] = S8_model

        parts['total'] = chi2
        return parts
    except Exception:
        return None

_lp_cache = {}

def log_posterior(theta):
    omm, H0, s8 = theta
    if not (OM_LO<omm<OM_HI and H0_LO<H0<H0_HI and S8_LO<s8<S8_HI):
        return -np.inf
    p = _chi2_decomposed(omm, H0, s8)
    if p is None:
        return -np.inf
    lp = -0.5*p['total']
    # cache for dashboard (keyed by rounded params to limit memory)
    key = (round(omm,6), round(H0,4), round(s8,5))
    if len(_lp_cache) > 500:
        _lp_cache.clear()
    _lp_cache[key] = p
    return lp

def get_cached_chi2(theta):
    """Return cached chi2 dict for theta if available, else recompute."""
    key = (round(theta[0],6), round(theta[1],4), round(theta[2],5))
    return _lp_cache.get(key) or _chi2_decomposed(theta[0], theta[1], theta[2])

# sanity checks
print('Sanity checks at test points:')
print(f'  {"omm":>8} {"H0":>6} {"theta*":>10} {"th_sig":>8} {"chi2_tot":>10}')
for omm_,H0_,s8_ in [(0.14415,69.39,0.7553),(0.150,70.,0.77),(0.155,71.,0.77),(0.160,69.,0.77)]:
    p = _chi2_decomposed(omm_,H0_,s8_)
    if p:
        print(f'  {omm_:>8.4f} {H0_:>6.2f} {p["th_val"]:>10.6f} {p["th_sig"]:>+8.2f} {p["total"]:>10.1f}')


Sanity checks at test points:
       omm     H0     theta*   th_sig   chi2_tot
    0.1442  69.39   0.010345   -10.71     1548.4
    0.1500  70.00   0.010420    +1.89     1440.0
    0.1550  71.00   0.010496   +14.54     1665.8
    0.1600  69.00   0.010484   +12.50     1569.8


In [ ]:
# Cell 7 — Checkpoint utilities + dashboard
import pickle, glob, time, shutil
from IPython.display import clear_output

def _save(path, obj):
    tmp = path+'.tmp'
    try:
        with open(tmp,'wb') as f: pickle.dump(obj,f,protocol=4); f.flush()
        os.replace(tmp,path); return True
    except Exception as e:
        print(f'  WARN save: {e}')
        try: os.remove(tmp)
        except: pass
        return False

def _load(path):
    try:
        with open(path,'rb') as f: return pickle.load(f)
    except:
        try: shutil.move(path,path+'.corrupt')
        except: pass
        return None

def _latest(pattern):
    files=sorted(glob.glob(os.path.join(CKPT_DIR,pattern)))
    while files:
        try:
            with open(files[-1],'rb') as f: pickle.load(f)
            return files[-1]
        except:
            try: shutil.move(files[-1],files[-1]+'.corrupt')
            except: pass
            files.pop()
    return None

def _cleanup(pat,keep=3):
    for f in sorted(glob.glob(os.path.join(CKPT_DIR,pat)))[:-keep]:
        try: os.remove(f)
        except: pass

W = 76
def _R(s): return f'\u2551  {s:<{W-4}}\u2551'

def dashboard(ch, lp, done, total, t0, phase, af=None):
    if len(lp) < 4: return
    recent = ch[max(0,len(ch)-max(len(ch)//2,200)):]
    ibest  = np.argmax(lp)
    bf     = ch[ibest]
    Om_bf  = bf[0]/(bf[1]/100.)**2
    Oneq_bf = 1. - Om_bf - (omega_r/(bf[1]/100.)**2)
    S8_bf  = bf[2]*np.sqrt(Om_bf/0.3)
    elap   = (time.time()-t0)/60.
    rate   = done/max(elap*60,1)
    eta    = (total-done)/max(rate/60,1e-6)
    pct    = done/total
    bar    = '\u2588'*int(30*pct)+'\u2591'*(30-int(30*pct))
    L=[f'\u2554{"\u2550"*W}\u2557']
    L.append(_R('EEC v10 (scale-free) \u2014 3 free params | \u03ba=\u03bb\u03c1_m, no z_c'))
    L.append(_R('Pantheon+ | BOSS DR12 | DESI DR1 | Planck \u03b8\u2605 (EEC geometry) | BBN \u03c9_b'))
    L.append(f'\u2560{"\u2550"*W}\u2563')
    L.append(_R(f'{phase}  [{bar}]  {pct*100:5.1f}%'))
    L.append(_R(f'Step {done:>6}/{total:<6}   {elap:.1f} min elapsed   ETA {eta:.0f} min'
                +(f'   accept={af:.3f}' if af else '')))
    L.append(f'\u2560{"\u2550"*W}\u2563')
    L.append(_R(f'RUNNING ESTIMATES  (last {len(recent)} samples)'))
    L.append(_R(f'{"Param":>12}  {"Mean":>9}  {"Std":>8}  {"16%":>9}  {"84%":>9}'))
    L.append(_R('\u2500'*62))
    for i,lab in enumerate(['\u03c9_m','H\u2080','\u03c3\u2088']):
        v=recent[:,i]
        L.append(_R(f'{lab:>12}  {np.mean(v):9.5f}  {np.std(v):8.5f}  '
                     f'{np.percentile(v,16):9.5f}  {np.percentile(v,84):9.5f}'))
    s_=min(100,len(recent))
    Om_r=np.mean([c[0]/(c[1]/100.)**2 for c in recent[-s_:]])
    Or_r=np.mean([omega_r/(c[1]/100.)**2 for c in recent[-s_:]])
    Or_r=omega_r/(np.mean(recent[-s_:,1])/100.)**2
    S8_r=np.mean([c[2]*np.sqrt(c[0]/(c[1]/100.)**2/0.3) for c in recent[-s_:]])
    L.append(_R('\u2500'*62))
    L.append(_R(f'{"\u03a9_m (deriv)":>12}  {Om_r:9.4f}'))
    L.append(_R(f'{"\u03a9_neq(deriv)":>12}  {1.-Om_r-Or_r:9.4f}'))
    L.append(_R(f'{"S\u2088 (deriv)":>12}  {S8_r:9.4f}'))
    L.append(f'\u2560{"\u2550"*W}\u2563')
    p=get_cached_chi2(bf)
    L.append(_R(f'BEST FIT  \u03c7\u00b2={-2*lp[ibest]:.1f}  '
                f'\u03c9_m={bf[0]:.5f}  H\u2080={bf[1]:.3f}  \u03c3\u2088={bf[2]:.4f}'))
    L.append(_R(f'          \u03a9_neq={Oneq_bf:.4f}  \u03a9_m={Om_bf:.4f}  S\u2088={S8_bf:.4f}'))
    if p:
        L.append(_R('\u2500'*62))
        L.append(_R(f'  \u03c7\u00b2 BREAKDOWN:'))
        L.append(_R(f'  Planck \u03b8\u2605  {p["th_val"]:.6f}  '
                     f'(obs {THETA_STAR_OBS})  '
                     f'\u03c7\u00b2={p["theta"]:6.1f}  '
                     f'({p["th_sig"]:+.2f}\u03c3)'))
        L.append(_R(f'  Pantheon+   {N_SN} SNe        \u03c7\u00b2={p["sn"]:7.1f}'))
        L.append(_R(f'  BOSS DR12                  \u03c7\u00b2={p["boss"]:7.2f}'))
        L.append(_R(f'  DESI iso                   \u03c7\u00b2={p["desi_iso"]:7.2f}'))
        L.append(_R(f'  DESI ani                   \u03c7\u00b2={p["desi_ani"]:7.2f}'))
        L.append(_R(f'  TOTAL                      \u03c7\u00b2={p["total"]:7.1f}'))
        L.append(_R(f'  (LCDM benchmark:           \u03c7\u00b2~ 1410)'))
    L.append(f'\u255a{"\u2550"*W}\u255d')
    clear_output(wait=True); print('\n'.join(L))

CKPT_BI   = os.path.join(CKPT_DIR,'burnin.pkl')
CKPT_DONE = os.path.join(CKPT_DIR,'final.pkl')
CKPT_PFMT = os.path.join(CKPT_DIR,'prod_{:07d}.pkl')
print('Utilities ready')


Utilities ready


In [ ]:
# Cell 8 — MCMC (3 free parameters)
import warnings; warnings.filterwarnings('ignore')
ndim = 3

chain = lp_chain = None
if os.path.exists(CKPT_DONE):
    r=_load(CKPT_DONE)
    if r and 'chain' in r:
        chain, lp_chain = r['chain'], r['lp_chain']
        print(f'Loaded final: {chain.shape}')

if chain is None:
    t0=time.time(); state=None; bi_done=0

    # ── resume burn-in ────────────────────────────────────────────────────
    if os.path.exists(CKPT_BI):
        r=_load(CKPT_BI)
        if r and 'state' in r:
            state=r['state']; bi_done=N_BURN
    if state is None:
        lb=_latest('burnin_step_*.pkl')
        if lb:
            r=_load(lb)
            if r: state=r['state']; bi_done=r.get('bi_done',0)

    # ── initialise walkers if needed ──────────────────────────────────────
    # Centre walkers at omm=0.150 (theta*-consistent region from scan)
    if state is None:
        bi_done=0
        p0c = np.array([0.150, 70.0, 0.77])
        p0s = np.array([0.010,  3.0, 0.06])
        valid=[]
        for _ in range(N_WALKERS*20):
            pw=p0c+p0s*np.random.randn(ndim)
            pw=np.clip(pw,[OM_LO+.005,H0_LO+1.,S8_LO+.02],
                          [OM_HI-.005,H0_HI-1.,S8_HI-.02])
            if np.isfinite(log_posterior(pw)):
                valid.append(pw)
                if len(valid)==N_WALKERS: break
        state=np.array(valid[:N_WALKERS])
        print(f'{len(state)} walkers initialised (centre: omm=0.150, H0=70.0)')

    # ── burn-in ───────────────────────────────────────────────────────────
    if bi_done < N_BURN:
        samp=emcee.EnsembleSampler(N_WALKERS,ndim,log_posterior)
        last_d=last_c=time.time()
        print(f'Burn-in: {N_BURN-bi_done} steps remaining')
        while bi_done < N_BURN:
            chunk=min(BATCH,N_BURN-bi_done)
            state=samp.run_mcmc(state,chunk,progress=False)
            bi_done+=chunk
            now=time.time()
            if now-last_d>=DISP_SECS or bi_done>=N_BURN:
                dashboard(samp.get_chain(flat=True),
                          samp.get_log_prob(flat=True),
                          bi_done,N_BURN,t0,'BURN-IN',
                          np.mean(samp.acceptance_fraction))
                last_d=now
            if now-last_c>=CKPT_SECS or bi_done>=N_BURN:
                _save(os.path.join(CKPT_DIR,f'burnin_step_{bi_done:05d}.pkl'),
                      {'state':state,'bi_done':bi_done})
                _cleanup('burnin_step_*.pkl',keep=2)
                last_c=now
        clear_output(wait=True)
        print(f'Burn-in done  accept={np.mean(samp.acceptance_fraction):.3f}')
        del samp
    _save(CKPT_BI,{'state':state})

    # ── production ────────────────────────────────────────────────────────
    lf=_latest('prod_*.pkl'); steps_done=0; chain_buf=np.empty((0,ndim)); lp_buf=np.empty(0)
    if lf:
        r=_load(lf)
        if r:
            state=r['state']; chain_buf=r['chain_buf']
            lp_buf=r['lp_buf']; steps_done=r['steps_done']

    if N_PROD-steps_done > 0:
        sampler=emcee.EnsembleSampler(N_WALKERS,ndim,log_posterior)
        last_d=last_c=time.time(); bc=[]; bl=[]
        print(f'Production: {N_PROD-steps_done} steps remaining')
        while steps_done < N_PROD:
            chunk=min(BATCH,N_PROD-steps_done)
            state=sampler.run_mcmc(state,chunk,progress=False)
            steps_done+=chunk
            bc.append(sampler.get_chain(flat=True)[-chunk*N_WALKERS:])
            bl.append(sampler.get_log_prob(flat=True)[-chunk*N_WALKERS:])
            now=time.time()
            if now-last_d>=DISP_SECS or steps_done>=N_PROD:
                bcc=np.concatenate([chain_buf]+bc)
                bll=np.concatenate([lp_buf]+bl)
                dashboard(bcc,bll,steps_done,N_PROD,t0,'PRODUCTION',
                          np.mean(sampler.acceptance_fraction))
                last_d=now
            if now-last_c>=CKPT_SECS or steps_done>=N_PROD:
                chain_buf=np.concatenate([chain_buf]+bc)
                lp_buf=np.concatenate([lp_buf]+bl)
                bc.clear(); bl.clear()
                _save(CKPT_PFMT.format(steps_done),
                      {'state':state,'chain_buf':chain_buf,
                       'lp_buf':lp_buf,'steps_done':steps_done})
                _cleanup('prod_*.pkl',keep=3)
                last_c=now
        if bc:
            chain_buf=np.concatenate([chain_buf]+bc)
            lp_buf=np.concatenate([lp_buf]+bl)

    chain,lp_chain=chain_buf,lp_buf
    _save(CKPT_DONE,{'chain':chain,'lp_chain':lp_chain})
    clear_output(wait=True)
    print(f'COMPLETE  {(time.time()-t0)/60.:.1f} min  samples={chain.shape}')


Production: 1940 steps remaining
╔════════════════════════════════════════════════════════════════════════════╗
║  EEC v10 (scale-free) — 3 free params | κ=λρ_m, no z_c                   ║
║  Pantheon+ | BOSS DR12 | DESI DR1 | Planck θ★ (EEC geometry) | BBN ω_b   ║
╠════════════════════════════════════════════════════════════════════════════╣
║  PRODUCTION  [██████████░░░░░░░░░░░░░░░░░░░░]   35.4%                    ║
║  Step   1062/3000     0.7 min elapsed   ETA 4562 min   accept=0.766      ║
╠════════════════════════════════════════════════════════════════════════════╣
║  RUNNING ESTIMATES  (last 16992 samples)                                 ║
║         Param       Mean       Std        16%        84%                 ║
║  ──────────────────────────────────────────────────────────────          ║
║           ω_m    0.15322   0.00102    0.15221    0.15420                 ║
║            H₀   68.57681   0.28265   68.30106   68.85814                 ║
║            σ₈    0.72546   0.03354 

In [ ]:
# Cell 9 — Results + tension analysis
ibest  = np.argmax(lp_chain)
bf     = chain[ibest]
Om_arr = np.array([c[0]/(c[1]/100.)**2 for c in chain])
Or_arr = np.array([omega_r/(c[1]/100.)**2 for c in chain])
Oc_arr = None  # scale-free: no Omega_c
S8_arr = chain[:,2]*np.sqrt(Om_arr/0.3)
OL_arr = 1.-Om_arr-Or_arr

Om_bf  = bf[0]/(bf[1]/100.)**2
Oneq_bf = 1. - Om_bf - (omega_r/(bf[1]/100.)**2)

Dc_bf,_,Ef_bf,_,_,_ = iterate_EEC(bf[0],bf[2],bf[1])
rd_bf  = rd(bf[0],bf[1])
age_v,_ = quad(lambda z:1./((1+z)*Ef_bf(z)),0,50,limit=300)
age_bf = age_v*3.0857e19/(bf[1]*3.1558e16)
th_bf  = compute_theta_star(bf[0],bf[1],Ef_bf)
th_sig = (th_bf-THETA_STAR_OBS)/THETA_STAR_ERR
pbf    = _chi2_decomposed(bf[0],bf[1],bf[2])

W2=64
print(f'\n{"="*W2}')
print(f'  EEC v10 RESULTS  (3 free params | \u03b8\u2605 constraint)')
print(f'  \u03c9_b={omega_b} (BBN fixed) | scale-free \u03ba=\u03bb\u03c1_m (no \u03a9_c)')
print(f'{"="*W2}')
print(f'  {"Param":>14}  {"Mean":>9}  {"Std":>8}  {"16%":>9}  {"84%":>9}')
print(f'  {"-"*58}')
labels = ['\u03c9_m','H\u2080','\u03c3\u2088']
for i,lab in enumerate(labels):
    v=chain[:,i]
    print(f'  {lab:>14}  {np.mean(v):9.5f}  {np.std(v):8.5f}  '
          f'{np.percentile(v,16):9.5f}  {np.percentile(v,84):9.5f}')
print(f'  {"-"*58}')
for lab,v in [('\u03a9_m',Om_arr),('\u03a9_neq',OL_arr),('S\u2088',S8_arr)]:
    print(f'  {lab:>14}  {np.mean(v):9.5f}  {np.std(v):8.5f}  '
          f'{np.percentile(v,16):9.5f}  {np.percentile(v,84):9.5f}')
print(f'\n  Best-fit:')
print(f'    \u03c9_m={bf[0]:.5f}  H\u2080={bf[1]:.3f}  \u03c3\u2088={bf[2]:.4f}')
print(f'    \u03a9_m={Om_bf:.4f}  \u03a9_neq={Oneq_bf:.4f}  r_d={rd_bf:.2f} Mpc  Age={age_bf:.2f} Gyr')
print(f'    S\u2088={bf[2]*np.sqrt(Om_bf/0.3):.4f}')
print(f'\n  \u03b8\u2605 (EEC geometry):')
print(f'    \u03b8\u2605_EEC  = {th_bf:.6f}')
print(f'    \u03b8\u2605_Planck= {THETA_STAR_OBS:.6f} \u00b1 {THETA_STAR_ERR:.6f}')
print(f'    Tension  = {th_sig:+.2f}\u03c3')
if pbf:
    print(f'\n  \u03c7\u00b2 decomposition:')
    print(f'    Planck \u03b8\u2605  {pbf["theta"]:8.2f}  ({pbf["th_sig"]:+.2f}\u03c3)')
    print(f'    Pantheon+   {pbf["sn"]:8.1f}  ({N_SN} SNe)')
    print(f'    BOSS DR12   {pbf["boss"]:8.2f}')
    print(f'    DESI iso    {pbf["desi_iso"]:8.2f}')
    print(f'    DESI ani    {pbf["desi_ani"]:8.2f}')
    print(f'    \u03c7\u00b2 TOTAL   {pbf["total"]:8.1f}')
    print(f'    LCDM bench  ~  1410')
H0m,H0e = np.mean(chain[:,1]),np.std(chain[:,1])
S8m,S8e = np.mean(S8_arr),np.std(S8_arr)
print(f'\n  Tension summary:')
print(f'    H\u2080 vs SH0ES (73.04\u00b11.04):  {abs(H0m-73.04)/np.sqrt(H0e**2+1.04**2):.1f}\u03c3')
print(f'    H\u2080 vs Planck (67.36\u00b10.54):  {abs(H0m-67.36)/np.sqrt(H0e**2+0.54**2):.1f}\u03c3')
print(f'    S\u2088 vs WL (0.762\u00b10.024):     {abs(S8m-0.762)/np.sqrt(S8e**2+0.024**2):.1f}\u03c3')
print(f'    S\u2088 vs Planck (0.832\u00b10.013):  {abs(S8m-0.832)/np.sqrt(S8e**2+0.013**2):.1f}\u03c3')
print(f'{"="*W2}')


In [ ]:
# Equation of state: w_eff(z) = -(4/3) f(z), and CPL-equivalent (w0, wa)
# Definition (Sec 10.2): rho_neq ~ (1+z)^3 D^4  =>  w_eff(z) = -(4/3) f(z), f = dln D/dln a.
from scipy.optimize import minimize as _minimize

_omm, _H0, _s8 = bf[0], bf[1], bf[2]
_Dc, _lam, _Ef, _Om, _Or, _ = iterate_EEC(_omm, _s8, _H0)
_Onq0 = 1. - _Om - _Or

# Growth rate f(z) = dln D/dln a, from fsig8: f(z) = fsig8(z, Dc) / Dc(z).
def f_growth(z):
    return fsig8(z, _Dc) / _Dc(z)

# f(0): evaluate the smooth z->0 limit by linear extrapolation (avoids the
# one-sided derivative at the a=1 ODE boundary).
def _f0():
    zs = np.array([0.005, 0.01, 0.02, 0.03])
    fs = np.array([f_growth(z) for z in zs])
    return float(np.polyfit(zs, fs, 1)[1])

f0 = _f0()
w_eff  = lambda z: -(4.0/3.0)*f_growth(z)
w_eff0 = -(4.0/3.0)*f0

# CPL-equivalent (w0, wa): fit w(a) = w0 + wa(1-a) to rho_DE(z) over 0 < z < 2.
def _Dn(z): return _Dc(z)/_s8
def _rho_neq(z): return _Onq0*(1+z)**3*_Dn(z)**4
def _ratio(z): return _rho_neq(z)/_rho_neq(0.)
def _cpl(z,w0,wa): return (1+z)**(3*(1+w0+wa))*np.exp(-3*wa*z/(1+z))
_zf = np.linspace(0,2,40); _tg = np.array([_ratio(z) for z in _zf])
_res = _minimize(lambda p: np.sum((np.log(_cpl(_zf,*p))-np.log(_tg))**2), [-1.,0.],
                 method='Nelder-Mead')
w0_cpl, wa_cpl = _res.x

print("="*58)
print("  EEC v10 - Equation of state  w_eff(z) = -(4/3) f(z)")
print("="*58)
print(f"  {'z':>5} {'f(z)':>9} {'w_eff(z)':>10}")
print(f"  {0.0:>5.1f} {f0:>9.4f} {w_eff0:>10.4f}")
for _z in [0.3, 0.5, 1.0, 2.0]:
    print(f"  {_z:>5.1f} {f_growth(_z):>9.4f} {w_eff(_z):>10.4f}")
print(f"\n  w_eff(0) = {w_eff0:.3f}   (f(0) = {f0:.3f})")
print(f"  CPL-equivalent:  w0 = {w0_cpl:.3f},  wa = {wa_cpl:.3f}")
print("="*58)


## Late-time fixed point: the exact attractor (v9 addition)

The coupled growth–expansion system is extended past $a=1$ with the **same physics** as the fitting engine (full $E(z)$ including radiation, self-consistent $\mu(z)$ iteration, growth ODE in $x=\ln a$). The system possesses a unique attracting fixed point, obtained in closed form from $6f_*^2+f_*-3=0$:

$$f_*=\frac{\sqrt{73}-1}{12}\approx 0.62867,\quad w_{\rm eff}^*=-\tfrac{4}{3}f_*=\frac{1-\sqrt{73}}{9}\approx-0.83822,$$
$$q_*=\tfrac{2}{3}-\tfrac{\sqrt{73}}{6}\approx-0.75733,\qquad H\propto a^{-\frac{3}{2}(1+w^*)}\approx a^{-0.2427}\to 0,\qquad \Omega_{\rm neq}\to 1.
$$

These values are **parameter-free**: independent of $\beta$, $\sigma_8$, and the initial amplitude (verified below for $\beta\in[4,66]$; agreement with the closed form to $2\times10^{-5}$, limited by grid resolution). $\Omega_{\rm neq}=0.674$ today is a position along this universal flow — the flow crosses $0.3<\Omega_{\rm neq}<0.9$ in $\sim$1 e-fold — so the present-day magnitude reduces exactly to the timing question, with the trajectory's normalization fixed by flatness (Section 25).


In [ ]:
# Cell — Late-time attractor of the EEC closure (full engine, extended past a=1)
# Same physics as iterate_EEC: full E(z) with radiation, self-consistent mu(z).
# Growth ODE solved in x = ln a from a0 = 1/(1+GROWTH_ZMAX) to a = e^15.
from scipy.integrate import solve_ivp as _sivp

_XMAX, _NPTS = 15.0, 1200
_FSTAR = (np.sqrt(73.)-1.)/12.

def _solve_growth_future(Om, Or, Onq0, Dgrid=None, x_grid=None):
    x0 = np.log(1./(1.+GROWTH_ZMAX))
    xg = np.linspace(x0, _XMAX, _NPTS) if x_grid is None else x_grid
    Dfun = (None if Dgrid is None else
            (lambda x, _l=np.log(np.clip(Dgrid,1e-300,None)), _x=xg:
             np.exp(np.interp(x, _x, _l))))
    i1 = np.argmin(np.abs(xg))
    def E2(x):
        a = np.exp(x); base = Om*a**-3 + Or*a**-4
        if Dfun is None: return base
        return base + Onq0*a**-3*Dfun(x)**4/Dfun(0.)**4
    def mu(x):
        if Dfun is None: return 1.
        a = np.exp(x)
        return 1. + (Onq0*a**-3*Dfun(x)**4/Dfun(0.)**4)/(Om*a**-3)
    def rhs(x, y):
        dx = 1e-4
        dlnE = 0.5*(np.log(E2(x+dx))-np.log(E2(x-dx)))/(2*dx)
        a = np.exp(x)
        return [y[1], -(2.+dlnE)*y[1] + 1.5*Om*mu(x)/(a**3*E2(x))*y[0]]
    a0 = np.exp(x0)
    s = _sivp(rhs, [x0,_XMAX], [a0,a0], t_eval=xg, rtol=1e-9, atol=1e-12)
    return xg, s.y[0]/s.y[0][i1], s.y[1]/s.y[0]

def iterate_EEC_future(omm, s8, H0, niter=25, tol=1e-9):
    """Self-consistent EEC solution to a = e^15. Returns x, D, f, Omega_neq."""
    h = H0/100.; Om, Or = omm/h**2, omega_r/h**2
    Onq0 = 1.-Om-Or
    xg, D, f = _solve_growth_future(Om, Or, Onq0)
    for _ in range(niter):
        xg, Dn, f = _solve_growth_future(Om, Or, Onq0, Dgrid=D, x_grid=xg)
        if np.max(np.abs(np.log(Dn)-np.log(D))) < tol: D = Dn; break
        D = Dn
    i1 = np.argmin(np.abs(xg)); a = np.exp(xg)
    rnq = Onq0*a**-3*D**4/D[i1]**4
    E2v = Om*a**-3 + Or*a**-4 + rnq
    return xg, D, f, rnq/E2v

# run at best fit (falls back to paper values if MCMC bf not in scope)
# use MCMC best-fit if available, else fall back to published best-fit
try:    _p = (bf[0], bf[2], bf[1])
except Exception: _p = (0.15322, 0.7333, 68.56)
xg, Dx, fx, Onqx = iterate_EEC_future(*_p)
_i1 = np.argmin(np.abs(xg))
print('='*62)
print('  EEC late-time attractor — full engine, extended past a=1')
print('='*62)
print(f'  exact: f* = (sqrt73-1)/12 = {_FSTAR:.6f},  w* = {(1-np.sqrt(73))/9:.6f},')
_wstar = (1-np.sqrt(73))/9
print(f'         q* = {2/3-np.sqrt(73)/6:.6f},  H ~ a^({-(3/2)*(1+_wstar):.4f}) -> 0,  Onq* = 1')
print(f'  today : f(0) = {fx[_i1]:.4f}   w_eff(0) = {-(4/3)*fx[_i1]:.4f}'
      f'   Onq(0) = {Onqx[_i1]:.4f}')
print(f'  a=e^15: f = {fx[-1]:.7f}   |f - f*| = {abs(fx[-1]-_FSTAR):.1e}')
print(f'  {"s8":>6} {"beta":>8} {"f(e^15)":>11} {"|f-f*|":>9}   [universality]')
for _s8 in (0.5, _p[1], 1.0):
    _x2,_D2,_f2,_O2 = iterate_EEC_future(_p[0], _s8, _p[2])
    _b = _beta_norm(_p[0]/(_p[2]/100.)**2, omega_r/(_p[2]/100.)**2, _s8)
    print(f'  {_s8:>6.3f} {_b:>8.2f} {_f2[-1]:>11.7f} {abs(_f2[-1]-_FSTAR):>9.1e}')
_j = np.argmin(np.abs(Onqx-0.674))
print(f'  flow crosses 0.3<Onq<0.9 in '
      f'{xg[np.argmin(np.abs(Onqx-0.9))]-xg[np.argmin(np.abs(Onqx-0.3))]:.2f} e-folds')
print('='*62)


In [ ]:
# Cell 10 — Corner plot
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import corner

fig = corner.corner(chain,
    labels=[r'$\omega_m$', r'$H_0$\ (km/s/Mpc)', r'$\sigma_8$'],
    quantiles=[0.16,0.5,0.84], show_titles=True,
    title_kwargs={'fontsize':11})
fig.suptitle(
    f'EEC v10  |  3 free params  |  $\\theta_\\star$ constraint  |  '
    f'$\\omega_b={omega_b}$ (BBN fixed)',
    y=1.02, fontsize=11)
fig.savefig('eec_v10_corner.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: eec_v10_corner.png')


In [ ]:
# Cell 11 — Save outputs
np.save('eec_v10_chain.npy', chain)
np.save('eec_v10_lp.npy',    lp_chain)

with open('eec_v10_results.txt','w') as f:
    f.write('EEC v10 Results\n')
    f.write('='*50+'\n')
    f.write(f'omega_b = {omega_b} (BBN, Cooke+2016 ApJ 830 148, fixed)\n')
    f.write('Scale-free formulation: kappa = beta*rho_m (no Omega_c)\n')
    f.write('3 free params: omega_m, H0, sigma_8\n')
    f.write('Planck theta* used (EEC geometry) — no LCDM omega_m prior\n\n')
    for i,lab in enumerate(['omega_m','H0','sigma_8']):
        v=chain[:,i]
        f.write(f'{lab}: {np.mean(v):.5f} +/- {np.std(v):.5f}\n')
    f.write('\nDerived:\n')
    for lab,v in [('Omega_m',Om_arr),
                  ('Omega_neq',OL_arr),('S8',S8_arr)]:
        f.write(f'{lab}: {np.mean(v):.5f} +/- {np.std(v):.5f}\n')
    f.write(f'\nBest fit: chi2={-2*lp_chain[ibest]:.2f}\n')
    f.write(f'omega_m={bf[0]:.5f} H0={bf[1]:.3f} sigma8={bf[2]:.4f}\n')
    f.write(f'theta*_EEC={th_bf:.6f} ({th_sig:+.2f} sigma from Planck)\n')
    f.write(f'H0={bf[1]:.3f}  S8={bf[2]*np.sqrt(Om_bf/0.3):.4f}  '
            f'rd={rd_bf:.2f} Mpc  Age={age_bf:.2f} Gyr\n')
    if pbf:
        f.write('\nChi2 decomposition:\n')
        for k in ['theta','sn','boss','desi_iso','desi_ani','total']:
            f.write(f'  {k}: {pbf[k]:.2f}\n')

print('Saved: eec_v10_results.txt, eec_v10_chain.npy, eec_v10_lp.npy')
try:
    from google.colab import files
    for fn in ['eec_v10_results.txt','eec_v10_corner.png']:
        try: files.download(fn)
        except: pass
except: pass
